# DSMEM collectives on H100 NVL — latency and bandwidth

Data: `../results/*.csv` produced per `../docs/EXPERIMENTS.md` (one H100 NVL, one thread-block cluster
per launch, `%globaltimer` stamps written by each CTA). Each row of `stamps.csv` is one timestamp of one
rank of one repetition.

**Setup** (fill in): GPU / driver / CUDA / Nsight Compute versions, clock state, commit hash — see
`../CLAUDE.md` *Tooling*.

**Definitions** (per launch): per-rank latency `L_r = exit_r − entry_r`; collective latency
`L = max_r exit_r − min_r entry_r`; skew `= max_r L_r − min_r L_r`; payload per CTA `P` (allreduce /
allgather: `M·B`; reducescatter / alltoall: `N·M·B`); `algbw = P / L`; `busbw = algbw · f` with
`f = 2(N−1)/N` for allreduce and `(N−1)/N` otherwise; per-SM injection bandwidth `S_r / L_r` with
`S_r` the bytes rank r sends (traffic model in `model.csv`).

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
R = Path('../results')
stamps = pd.read_csv(R / 'stamps.csv')
model  = pd.read_csv(R / 'model.csv')
ncu    = pd.read_csv(R / 'ncu.csv') if (R / 'ncu.csv').exists() else None
BUS_F = {'allreduce': lambda n: 2*(n-1)/n, 'allgather': lambda n: (n-1)/n,
         'reducescatter': lambda n: (n-1)/n, 'alltoall': lambda n: (n-1)/n}
keys = ['file','op','alg','N','M','elem_bytes','payload_bytes','rep']

In [ ]:
# per-rank and per-launch latencies
w = stamps.pivot_table(index=keys+['rank','smid'], columns='stamp', values='t_ns').reset_index()
w['L_r'] = w['exit'] - w['entry']
per_launch = (w.groupby(keys)
               .agg(L=('exit','max'), t0=('entry','min'), Lr_max=('L_r','max'), Lr_min=('L_r','min'))
               .reset_index())
per_launch['L'] = per_launch['L'] - per_launch['t0']
per_launch['skew'] = per_launch['Lr_max'] - per_launch['Lr_min']
per_launch['algbw_GBs'] = per_launch['payload_bytes'] / per_launch['L']          # bytes/ns = GB/s
per_launch['busbw_GBs'] = per_launch.apply(lambda r: r.algbw_GBs * BUS_F[r.op](r.N), axis=1)
cfg = (per_launch.groupby(['file','op','alg','N','M','elem_bytes','payload_bytes'])
         .agg(L_med=('L','median'), L_p10=('L', lambda s: s.quantile(.1)), L_p90=('L', lambda s: s.quantile(.9)),
              skew_med=('skew','median'), algbw=('algbw_GBs','median'), busbw=('busbw_GBs','median'))
         .reset_index())
cfg.head()

## 1. Collective latency vs cluster size (payload 4 KB per CTA)

In [ ]:
P0 = 4096
ops = ['allreduce','allgather','reducescatter','alltoall']
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
for ax, op in zip(axes, ops):
    d = cfg[(cfg.op == op) & (cfg.payload_bytes == P0)]
    for alg, g in d.groupby('alg'):
        g = g.sort_values('N'); ax.plot(g.N, g.L_med, marker='o', label=alg)
        ax.fill_between(g.N, g.L_p10, g.L_p90, alpha=.15)
    ax.set(title=op, xlabel='cluster size N', xscale='log', yscale='log', xticks=[2,4,8,16], xticklabels=[2,4,8,16])
    ax.legend(fontsize=7)
axes[0].set_ylabel('collective latency L (ns), median, p10–p90')
plt.tight_layout()

## 2. Latency vs payload (N = 8)

In [ ]:
N0 = 8
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
for ax, op in zip(axes, ops):
    d = cfg[(cfg.op == op) & (cfg.N == N0)]
    for alg, g in d.groupby('alg'):
        g = g.sort_values('payload_bytes'); ax.plot(g.payload_bytes, g.L_med, marker='o', label=alg)
    ax.set(title=f'{op}, N={N0}', xlabel='payload per CTA (B)', xscale='log', yscale='log'); ax.legend(fontsize=7)
axes[0].set_ylabel('collective latency L (ns)')
plt.tight_layout()

## 3. Bus bandwidth vs payload (N = 8)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
for ax, op in zip(axes, ops):
    d = cfg[(cfg.op == op) & (cfg.N == N0)]
    for alg, g in d.groupby('alg'):
        g = g.sort_values('payload_bytes'); ax.plot(g.payload_bytes, g.busbw, marker='o', label=alg)
    ax.set(title=f'{op}, N={N0}', xlabel='payload per CTA (B)', xscale='log'); ax.legend(fontsize=7)
axes[0].set_ylabel('bus bandwidth (GB/s)')
plt.tight_layout()

## 4. Per-rank timeline (entry, rounds, exit) for one configuration per op
Each horizontal line is one rank (labelled with its SM id); markers are the round stamps. The slowest rank is the critical path.

In [ ]:
def timeline(ax, file, N, payload, rep=None):
    d = w[(w.file == file) & (w.N == N) & (w.payload_bytes == payload)]
    if rep is None: rep = d.rep.iloc[len(d) // 2]
    d = d[d.rep == rep]
    t0 = d['entry'].min()
    rounds = sorted([c for c in d.columns if str(c).startswith('round')], key=lambda c: int(c[5:]))
    for _, r in d.sort_values('rank').iterrows():
        xs = [r['entry'] - t0] + [r[c] - t0 for c in rounds if not np.isnan(r[c])] + [r['exit'] - t0]
        ax.plot(xs, [r['rank']] * len(xs), marker='|', ms=12)
        ax.text(xs[-1], r['rank'], f"  SM{int(r['smid'])}", va='center', fontsize=7)
    ax.set(title=f'{file}  N={N}  {payload} B', xlabel='ns since first entry', ylabel='rank')
examples = [('allreduce__ring__nccl.cu', 8, 4096), ('allgather__recursive_doubling__clusterfusion.cu', 8, 4096),
            ('reducescatter__direct_push__generic.cu', 8, 4096), ('alltoall__bruck__mpich.cu', 8, 4096)]
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
for ax, (f, n, p) in zip(axes.ravel(), examples): timeline(ax, f, n, p)
plt.tight_layout()

## 5. Per-SM injection bandwidth (N = 16, 16 KB per CTA)
Bytes each rank sends (traffic model) divided by its own latency; one box per algorithm, points are ranks × reps.

In [ ]:
m = model.rename(columns={'bytes_sent': 'S_r'})[['file','N','M','rank','S_r']]
ww = w.merge(m, on=['file','N','M','rank'], how='left')
ww['inj_GBs'] = ww['S_r'] / ww['L_r']
d = ww[(ww.N == 16) & (ww.payload_bytes == 16384)]
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
for ax, op in zip(axes, ops):
    dd = d[d.op == op]
    groups = [g.inj_GBs.dropna().values for _, g in dd.groupby('alg')]
    ax.boxplot(groups, labels=[a for a, _ in dd.groupby('alg')], showfliers=False)
    ax.set(title=op); ax.tick_params(axis='x', rotation=60, labelsize=7)
axes[0].set_ylabel('per-SM injection bandwidth (GB/s)')
plt.tight_layout()

## 6. Measured DSMEM traffic (Nsight Compute) vs traffic model
Per-CTA bytes: ncu kernel-wide counter / N against the docstring formula. Points off the diagonal mean the model or the implementation is wrong.

In [ ]:
if ncu is not None:
    tot = model.groupby(['file','N','M'])[['bytes_sent','bytes_recv']].sum().reset_index()
    tot['model_bytes_per_cta'] = (tot.bytes_sent + tot.bytes_recv) / tot.N
    meas = ncu[ncu.metric.str.contains('dsmem|cluster', case=False)].groupby(['file','N','M'])['value'].sum().reset_index()
    meas['ncu_bytes_per_cta'] = meas['value'] / meas['N']
    j = tot.merge(meas, on=['file','N','M'])
    fig, ax = plt.subplots(figsize=(6, 6))
    for alg, g in j.groupby(j.file.str.split('__').str[1]):
        ax.scatter(g.model_bytes_per_cta, g.ncu_bytes_per_cta, label=alg, s=14)
    lim = [j[['model_bytes_per_cta','ncu_bytes_per_cta']].min().min(), j[['model_bytes_per_cta','ncu_bytes_per_cta']].max().max()]
    ax.plot(lim, lim, 'k--', lw=1); ax.set(xscale='log', yscale='log', xlabel='model bytes / CTA', ylabel='ncu bytes / CTA')
    ax.legend(fontsize=6, ncol=2)
else:
    print('results/ncu.csv not found')

## Summary table
Median collective latency (ns) at 4 KB per CTA, rows = algorithm, columns = N.

In [ ]:
pd.set_option('display.width', 200)
(cfg[cfg.payload_bytes == P0].pivot_table(index=['op','alg'], columns='N', values='L_med').round(0))